# 🔍 BIST 100 Bronze Layer Data Exploration & Analysis

Welcome to the **MDK Trading Oracle** Bronze Exploration Notebook.

### Objectives
1. Connect to local **DuckDB** in concurrent read-only mode (`read_only=True`).
2. Inspect the **Bronze Tables** loaded directly from the raw data dump (36.8+ million tick-by-tick trades across March 2026).
3. Investigate raw trade structures, broker behaviors (focusing on institutional flows like `MLB` / Bank of America), volume distributions, and price dynamics.
4. Experiment with transformations and aggregations to design our **Silver Layer**.

## 1. Connect to DuckDB & Initialize Environment

In [ ]:
from pathlib import Path
import duckdb
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

# Connect with read_only=True to prevent file locks during concurrent analysis
db_path = Path("../data/database/mdk_oracle.duckdb")
conn = duckdb.connect(str(db_path), read_only=True)

print(f" Connected to DuckDB: {db_path.resolve()}")

## 2. Inspect Bronze Tables & Metadata

In [ ]:
# List all tables in the database
tables_df = conn.execute("SHOW TABLES;").pl()
print("Available Tables:")
tables_df

In [ ]:
# Inspect Bronze schema & row counts
counts_df = conn.execute("""
    SELECT 
        'bronze_raw_trades' AS table_name, 
        COUNT(*) AS total_rows, 
        MIN(timestamp) AS min_time, 
        MAX(timestamp) AS max_time 
    FROM bronze_raw_trades
    UNION ALL
    SELECT 
        'bronze_brokers' AS table_name, 
        COUNT(*) AS total_rows, 
        NULL AS min_time, 
        NULL AS max_time 
    FROM bronze_brokers
    UNION ALL
    SELECT 
        'bronze_instruments' AS table_name, 
        COUNT(*) AS total_rows, 
        NULL AS min_time, 
        NULL AS max_time 
    FROM bronze_instruments;
""").pl()

counts_df

## 3. Sample Raw Trades Structure

In [ ]:
# Preview first 10 rows of tick-by-tick trades
sample_trades = conn.execute("""
    SELECT 
        timestamp,
        symbol,
        price,
        volume,
        ROUND(price * volume, 2) AS turnover_tl,
        buyer_broker_id,
        seller_broker_id,
        raw_source
    FROM bronze_raw_trades
    LIMIT 10;
""").pl()

sample_trades

## 4. Daily Market Activity (Trading Days in March 2026)

In [ ]:
daily_summary = conn.execute("""
    SELECT 
        CAST(timestamp AS DATE) AS trade_date,
        COUNT(*) AS trade_count,
        COUNT(DISTINCT symbol) AS active_symbols,
        ROUND(SUM(volume), 0) AS total_lots_traded,
        ROUND(SUM(price * volume), 2) AS total_market_turnover_tl
    FROM bronze_raw_trades
    GROUP BY trade_date
    ORDER BY trade_date ASC;
""").pl()

daily_summary

In [ ]:
# Plot Daily Turnover
fig_turnover = px.bar(
    daily_summary.to_pandas(),
    x="trade_date",
    y="total_market_turnover_tl",
    title="BIST Total Market Daily Turnover (TL) - March 2026",
    labels={"trade_date": "Date", "total_market_turnover_tl": "Turnover (TL)"},
    template="plotly_dark",
    color_discrete_sequence=["#00d2d3"]
)
fig_turnover.update_layout(height=450)
fig_turnover.show()

## 5. Top Traded Symbols by Volume & Turnover

In [ ]:
top_symbols = conn.execute("""
    SELECT 
        symbol,
        COUNT(*) AS total_trades,
        ROUND(SUM(volume), 0) AS total_volume_lots,
        ROUND(SUM(price * volume), 2) AS total_turnover_tl,
        ROUND(AVG(price), 2) AS avg_price,
        MIN(price) AS min_price,
        MAX(price) AS max_price
    FROM bronze_raw_trades
    GROUP BY symbol
    ORDER BY total_turnover_tl DESC
    LIMIT 15;
""").pl()

top_symbols

## 6. Broker Activity & Market Share Overview
Let's see which brokers dominate buy, sell, and turnover volumes across the entire month.

In [ ]:
broker_activity = conn.execute("""
    WITH buy_stats AS (
        SELECT 
            buyer_broker_id AS broker_id,
            SUM(volume) AS buy_volume,
            SUM(price * volume) AS buy_turnover_tl,
            COUNT(*) AS buy_trades
        FROM bronze_raw_trades
        GROUP BY buyer_broker_id
    ),
    sell_stats AS (
        SELECT 
            seller_broker_id AS broker_id,
            SUM(volume) AS sell_volume,
            SUM(price * volume) AS sell_turnover_tl,
            COUNT(*) AS sell_trades
        FROM bronze_raw_trades
        GROUP BY seller_broker_id
    )
    SELECT 
        COALESCE(b.broker_name, COALESCE(bs.broker_id, ss.broker_id)) AS broker_name,
        COALESCE(bs.broker_id, ss.broker_id) AS broker_id,
        b.category,
        ROUND(COALESCE(bs.buy_turnover_tl, 0), 2) AS total_buy_tl,
        ROUND(COALESCE(ss.sell_turnover_tl, 0), 2) AS total_sell_tl,
        ROUND(COALESCE(bs.buy_turnover_tl, 0) - COALESCE(ss.sell_turnover_tl, 0), 2) AS net_flow_tl,
        ROUND(COALESCE(bs.buy_turnover_tl, 0) + COALESCE(ss.sell_turnover_tl, 0), 2) AS total_turnover_tl
    FROM buy_stats bs
    FULL OUTER JOIN sell_stats ss ON bs.broker_id = ss.broker_id
    LEFT JOIN bronze_brokers b ON COALESCE(bs.broker_id, ss.broker_id) = b.broker_id
    ORDER BY total_turnover_tl DESC
    LIMIT 20;
""").pl()

broker_activity

## 7. Deep Dive: Bank of America (`MLB`) Daily Net Flow by Symbol

In [ ]:
# Calculate BofA (MLB) daily buys, sells, net flow, and VWAP per symbol
bofa_symbol_daily = conn.execute("""
    WITH bofa_buys AS (
        SELECT 
            CAST(timestamp AS DATE) AS trade_date,
            symbol,
            SUM(volume) AS buy_volume,
            SUM(price * volume) AS buy_tl
        FROM bronze_raw_trades
        WHERE buyer_broker_id = 'MLB'
        GROUP BY trade_date, symbol
    ),
    bofa_sells AS (
        SELECT 
            CAST(timestamp AS DATE) AS trade_date,
            symbol,
            SUM(volume) AS sell_volume,
            SUM(price * volume) AS sell_tl
        FROM bronze_raw_trades
        WHERE seller_broker_id = 'MLB'
        GROUP BY trade_date, symbol
    )
    SELECT 
        COALESCE(b.trade_date, s.trade_date) AS trade_date,
        COALESCE(b.symbol, s.symbol) AS symbol,
        COALESCE(b.buy_volume, 0) AS buy_volume,
        COALESCE(s.sell_volume, 0) AS sell_volume,
        COALESCE(b.buy_volume, 0) - COALESCE(s.sell_volume, 0) AS net_volume,
        ROUND(COALESCE(b.buy_tl, 0), 2) AS buy_tl,
        ROUND(COALESCE(s.sell_tl, 0), 2) AS sell_tl,
        ROUND(COALESCE(b.buy_tl, 0) - COALESCE(s.sell_tl, 0), 2) AS net_tl
    FROM bofa_buys b
    FULL OUTER JOIN bofa_sells s ON b.trade_date = s.trade_date AND b.symbol = s.symbol
    WHERE symbol IN ('THYAO', 'AKBNK', 'GARAN', 'EREGL', 'TUPRS')
    ORDER BY trade_date ASC, symbol ASC;
""").pl()

bofa_symbol_daily.head(20)

### Plot BofA Cumulative Flow on THYAO

In [ ]:
thyao_bofa = bofa_symbol_daily.filter(pl.col("symbol") == "THYAO").with_columns(
    pl.col("net_tl").cum_sum().alias("cum_net_tl")
)

fig_bofa = go.Figure()

fig_bofa.add_trace(go.Bar(
    x=thyao_bofa["trade_date"].to_list(),
    y=thyao_bofa["net_tl"].to_list(),
    name="Daily Net TL",
    marker=dict(color=["#10ac84" if v > 0 else "#ee5253" for v in thyao_bofa["net_tl"]]),
    opacity=0.7
))

fig_bofa.add_trace(go.Scatter(
    x=thyao_bofa["trade_date"].to_list(),
    y=thyao_bofa["cum_net_tl"].to_list(),
    name="Cumulative Net Flow (TL)",
    line=dict(color="#feca57", width=3)
))

fig_bofa.update_layout(
    title="Bank of America (MLB) - THYAO Daily & Cumulative Net Flow (March 2026)",
    xaxis_title="Date",
    yaxis_title="TL Amount",
    template="plotly_dark",
    height=500
)
fig_bofa.show()

## 8. Intraday Trade Arrival Distribution
Let's examine trade distribution by hour of day (10:00 to 18:00 session) to understand intraday liquidity patterns.

In [ ]:
hourly_dist = conn.execute("""
    SELECT 
        EXTRACT(HOUR FROM timestamp) AS hour_of_day,
        COUNT(*) AS trade_count,
        ROUND(SUM(price * volume), 2) AS hourly_turnover_tl
    FROM bronze_raw_trades
    WHERE timestamp IS NOT NULL
    GROUP BY hour_of_day
    ORDER BY hour_of_day ASC;
""").pl()

hourly_dist

In [ ]:
fig_hourly = px.bar(
    hourly_dist.to_pandas(),
    x="hour_of_day",
    y="trade_count",
    title="BIST Trade Frequency by Hour of Day",
    labels={"hour_of_day": "Hour of Day (UTC/Local)", "trade_count": "Total Number of Trades"},
    template="plotly_dark",
    color_discrete_sequence=["#54a0ff"]
)
fig_hourly.update_layout(height=400)
fig_hourly.show()

## 💡 Next Steps: Building the Silver Layer

Based on our findings in this Bronze exploration:
1. **`silver_daily_broker_summary`**: Aggregate daily buy/sell volumes, turnover, and buy/sell VWAP per (date, symbol, broker).
2. **`silver_market_daily`**: Aggregate daily OHLCV and market turnover per (date, symbol).
3. **`silver_broker_transactions`**: Clean, deduplicated intraday transactions with broker name enrichment.

We can build these Silver tables incrementally in the next notebook (`02_silver_transformations.ipynb`) or add ETL transformation functions under `src/mdk_trading_oracle/`!